In [ ]:
%run "./00_CC_Mapping_Setup.ipynb" 

In [ ]:
%sql
/* ===================================================================================
   METRIC: TP04 - TP Sole Source
   
   BUSINESS LOGIC: 
   - Denominator: N/A (Numerical Count).
   - Numerator: No of the unit's active third party vendors who are single source 
     AND located in high risk jurisdictions.
   - Rule 1: Use Single Source file to bridge Engagement ID to ThirdPartyName.
   - Rule 2: Col D, M, N: if any have high risk country, counted as 1.
   - Rule 3: Col K & L format can be comma-separated. Expand rows and remove duplicates.
   
   ARCHITECTURE SUMMARY:
   1. MASTER AU ANCHOR: Output strictly controlled by the filtered Master AU list.
   2. SINGLE SOURCE BRIDGE: Filters published_contracts for 'Single Source'.
   3. FLATTEN LOBs: Uses LATERAL VIEW explode(split(...)) to expand comma-separated 
      lob_using and owning_lob strings into individual evaluation rows.
   4. SAFE MAPPING: Joins expanded LOBs using standard LIKE syntax.
   5. DEDUPLICATION: Uses COUNT(DISTINCT EngagementNumber) grouped by AU_ID.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Mapped_AUs AS (
    -- 2. Grab valid TPRM Mapping Strings (Treating Assessable_Unit_ID as a Name String)
    SELECT DISTINCT 
        TRIM(Assessable_Unit_ID) AS Mapping_AU_Name,
        TRIM(TPRM_Units) AS TPRM_Units
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
    WHERE Assessable_Unit_ID IS NOT NULL
      AND NULLIF(TRIM(TPRM_Units), '') IS NOT NULL
),

High_Risk_Countries AS (
    -- 3. Grab high-risk jurisdictions
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Single_Source_Engagements AS (
    -- 4. Bridge the true Engagement ID from the Single Source file
    SELECT DISTINCT TRIM(ThirdPartyName) AS True_Engagement_ID
    FROM hive_metastore.ra_adido_2025.published_contrcats_19_dec_2025
    WHERE UPPER(TRIM(EngagementNumber)) = 'SINGLE SOURCE'
      AND ThirdPartyName IS NOT NULL
),

Filtered_Engagements AS (
    -- 5. Combine TP base data, Single Source bridge, and High Risk filter
    SELECT 
        tp.EngagementNumber,
        tp.owning_lob,
        tp.lob_using
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 tp
    INNER JOIN Single_Source_Engagements s
        ON TRIM(tp.EngagementNumber) = s.True_Engagement_ID
    LEFT JOIN High_Risk_Countries h1 ON UPPER(TRIM(tp.location_of_tp)) = h1.CountryName
    LEFT JOIN High_Risk_Countries h2 ON UPPER(TRIM(tp.country_prod_serv_originates)) = h2.CountryName
    LEFT JOIN High_Risk_Countries h3 ON UPPER(TRIM(tp.Td_Countries_Impacted)) = h3.CountryName
    WHERE h1.CountryName IS NOT NULL
       OR h2.CountryName IS NOT NULL
       OR h3.CountryName IS NOT NULL
       OR (COALESCE(TRIM(tp.location_of_tp), '') = ''
           AND COALESCE(TRIM(tp.country_prod_serv_originates), '') = ''
           AND COALESCE(TRIM(tp.Td_Countries_Impacted), '') = '')
),

Flattened_LOBs AS (
    -- 6. FLATTEN RULE: Expand comma-separated values into individual rows
    SELECT 
        EngagementNumber, 
        TRIM(exploded_lob) AS Expanded_LOB
    FROM Filtered_Engagements
    LATERAL VIEW explode(split(owning_lob, ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
    
    UNION ALL
    
    SELECT 
        EngagementNumber, 
        TRIM(exploded_lob) AS Expanded_LOB
    FROM Filtered_Engagements
    LATERAL VIEW explode(split(lob_using, ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
),

Mapped_Engagements AS (
    -- 7. Map expanded rows to the string-based Mapping AU Name using SAFE LIKE
    SELECT DISTINCT 
        f.EngagementNumber,
        map.Mapping_AU_Name
    FROM Flattened_LOBs f
    INNER JOIN Mapped_AUs map
        ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
),

Engagements_By_AU AS (
    -- 8. Bridge the Mapping Name to the Master List to get the true Numeric ID
    SELECT 
        mast.BusinessID,
        COUNT(DISTINCT m.EngagementNumber) AS High_Risk_Count
    FROM Mapped_Engagements m
    INNER JOIN Master_AUs mast
        ON UPPER(TRIM(m.Mapping_AU_Name)) = UPPER(TRIM(mast.AU_Name))
    GROUP BY mast.BusinessID
)

-- 9. Final Output
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'TP04' AS QuestionID,               
    COALESCE(CAST(e.High_Risk_Count AS STRING), '0') AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Engagements_By_AU e 
    ON a.BusinessID = e.BusinessID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: TP04 - Mapping & Flattening Verification
   
   PURPOSE: Prints the raw Col K and Col L strings alongside the flattened LOBs, 
   the mapping table strings, and the attempted Master AU lookup to troubleshoot
   why AUs are rolling up to 0.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS Master_Numeric_ID,
        TRIM(Business_Operational_Unit_Name) AS Master_AU_Name
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
),

High_Risk_Countries AS (
    SELECT UPPER(TRIM(Jurisdiction)) AS CountryName
    FROM hive_metastore.ra_adido_2025.td_country_risk_rating_abac 
    WHERE TRIM(FinalABACRating) = 'High'
),

Single_Source_Engagements AS (
    SELECT DISTINCT TRIM(ThirdPartyName) AS True_Engagement_ID
    FROM hive_metastore.ra_adido_2025.published_contrcats_19_dec_2025
    WHERE UPPER(TRIM(EngagementNumber)) = 'SINGLE SOURCE'
      AND ThirdPartyName IS NOT NULL
),

Mapped_AUs AS (
    -- Selecting EVERYTHING from the mapping table so we can see all available columns
    SELECT *
    FROM hive_metastore.ra_adido_2025.third_party_unit_mapping
),

Filtered_Engagements AS (
    SELECT 
        tp.EngagementNumber,
        tp.owning_lob,
        tp.lob_using,
        tp.location_of_tp,
        tp.country_prod_serv_originates,
        tp.Td_Countries_Impacted
    FROM hive_metastore.ra_adido_2025.abac_request_paul_v3 tp
    INNER JOIN Single_Source_Engagements s
        ON TRIM(tp.EngagementNumber) = s.True_Engagement_ID
    LEFT JOIN High_Risk_Countries h1 ON UPPER(TRIM(tp.location_of_tp)) = h1.CountryName
    LEFT JOIN High_Risk_Countries h2 ON UPPER(TRIM(tp.country_prod_serv_originates)) = h2.CountryName
    LEFT JOIN High_Risk_Countries h3 ON UPPER(TRIM(tp.Td_Countries_Impacted)) = h3.CountryName
    WHERE h1.CountryName IS NOT NULL
       OR h2.CountryName IS NOT NULL
       OR h3.CountryName IS NOT NULL
       OR (COALESCE(TRIM(tp.location_of_tp), '') = ''
           AND COALESCE(TRIM(tp.country_prod_serv_originates), '') = ''
           AND COALESCE(TRIM(tp.Td_Countries_Impacted), '') = '')
),

Flattened_LOBs AS (
    -- Pass the raw columns completely untouched while exploding the rows
    SELECT 
        EngagementNumber, 
        owning_lob AS Raw_owning_lob,
        lob_using AS Raw_lob_using,
        TRIM(exploded_lob) AS Expanded_LOB,
        'owning_lob' AS Source_Column
    FROM Filtered_Engagements
    LATERAL VIEW explode(split(owning_lob, ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
    
    UNION ALL
    
    SELECT 
        EngagementNumber, 
        owning_lob AS Raw_owning_lob,
        lob_using AS Raw_lob_using,
        TRIM(exploded_lob) AS Expanded_LOB,
        'lob_using' AS Source_Column
    FROM Filtered_Engagements
    LATERAL VIEW explode(split(lob_using, ',')) AS exploded_lob
    WHERE exploded_lob IS NOT NULL AND TRIM(exploded_lob) != ''
)

SELECT DISTINCT
    f.EngagementNumber,
    
    -- Exposing the exact original strings
    f.Raw_owning_lob,
    f.Raw_lob_using,
    
    -- Showing the exploded chunks
    f.Expanded_LOB,
    f.Source_Column,
    
    -- Showing what the mapping table matched on, and what it thinks the ID is
    map.TPRM_Units AS Matched_Mapping_String,
    map.Assessable_Unit_ID AS Raw_Mapping_Table_ID_Column,
    
    -- Attempting to bridge the Mapping Table's ID to the Master AU List via the Name
    mast.Master_Numeric_ID AS Matched_Master_Numeric_ID,
    mast.Master_AU_Name
    
FROM Flattened_LOBs f
-- Standard LIKE mapping
LEFT JOIN Mapped_AUs map
    ON UPPER(f.Expanded_LOB) LIKE '%' || UPPER(map.TPRM_Units) || '%'
    
-- Attempt to bridge the Mapping Table's string ID to the Master AU List Name
LEFT JOIN Master_AUs mast
    ON UPPER(TRIM(map.Assessable_Unit_ID)) = UPPER(mast.Master_AU_Name)
    
ORDER BY f.EngagementNumber;